# Notebook 21: ML System Design for Frontier Lab Interviews

**Goal**: Master the systems thinking that frontier AI labs (Anthropic, OpenAI, DeepMind, Meta FAIR) test in their ML system design interviews. This covers distributed training, inference optimization, and end-to-end design problems.

**Format**: Self-quiz first, code before theory, "Why does this work?", flashcards at end, paper guide at end.

---

## 1. Self-Quiz: Test Yourself First

Before reading anything, try to answer these from memory. Write your answers in the cells below, then check against the material that follows.

1. **What are the 3 types of model parallelism?** (Not data parallelism -- actual model parallelism strategies)
2. **What is PagedAttention?** Why was it invented? What problem does it solve?
3. **What is speculative decoding?** How does it achieve speedup without quality loss?
4. **What are DeepSpeed ZeRO stages?** What does each stage shard, and what are the memory savings?

In [ ]:
# YOUR ANSWERS (fill in before reading ahead)

q1_model_parallelism_types = """
1. 
2. 
3. 
"""

q2_paged_attention = """

"""

q3_speculative_decoding = """

"""

q4_deepspeed_zero = """
Stage 1: 
Stage 2: 
Stage 3: 
"""

print("Answers recorded. Now read on and check yourself.")

---
## 2. Setup

In [ ]:
!pip install torch matplotlib numpy -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
import math
from typing import Optional, Tuple, Dict, List

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 3. The System Design Framework

### The Frontier Lab Mindset

> **"GPU cluster, inference batch, model binary -- the model is a black box. Design the system around it."**

At frontier labs, the model is often treated as a given. Your job is to design the **infrastructure** that makes it trainable, serveable, and reliable at scale. This is fundamentally different from traditional ML system design where the model architecture is the focus.

### Interview Format (Anthropic-style)

- **Duration**: 50-55 minutes
- **Format**: Open-ended prompt ("Design a system for X")
- **Who drives**: YOU. The interviewer gives a problem, you drive the design. They nudge if you go off track.
- **What they want**: Structured thinking, trade-off awareness, depth in specific areas when probed.

### The Framework: 7 Steps

Use this framework for every ML system design question:

| Step | Time | What to Cover |
|------|------|---------------|
| **1. Requirements** | 5 min | Functional + non-functional. Scale numbers. SLOs. |
| **2. Architecture** | 8 min | High-level components. Data stores. Queues. Services. |
| **3. Data Flow** | 5 min | How data moves through the system. Batch vs streaming. |
| **4. Training Pipeline** | 10 min | Data prep, distributed training, evaluation, checkpointing. |
| **5. Serving** | 10 min | Inference optimization, batching, caching, load balancing. |
| **6. Monitoring** | 5 min | Metrics, alerts, drift detection, A/B testing. |
| **7. Cost** | 5 min | GPU hours, storage, bandwidth. Optimization strategies. |

**Pro tip**: Always start by asking clarifying questions about scale. "Are we talking about 100 QPS or 100K QPS?" changes everything.

**Insider Tip:** Anthropic's system design interviews treat the model as a BLACK BOX. You design the infrastructure around it: queuing, batching, routing, error handling, scale. Don't waste time discussing model architecture -- focus on the system.

**Insider Tip:** The most impressive thing in a system design interview is showing COST AWARENESS. Calculate: 'At 10K QPS with 500 input + 200 output tokens average, using A100s at $X/hr, the monthly cost is $Y. Here's how we optimize...' Numbers impress.

**Insider Tip:** Know these numbers cold: A100 has 80GB HBM, H100 has 80GB HBM with 3TB/s bandwidth, a 70B model in fp16 needs ~140GB (2 A100s), in int4 needs ~35GB (1 A100). Inference throughput: ~100-500 tokens/sec per A100 for a 70B model depending on batch size.

---
## 4. Distributed Training Deep Dive

### The Landscape

Training frontier models requires distributing work across hundreds or thousands of GPUs. There are fundamentally different strategies, each with distinct trade-offs.

```
Distributed Training Strategies
================================

Data Parallel          Model Parallel               Expert Parallel
  (DDP/FSDP)       (Tensor / Pipeline)               (MoE routing)
      |                    |                               |
  Replicate model     Split model                    Route experts
  Split data          across GPUs                    to different GPUs
      |                    |                               |
  AllReduce           Point-to-point                 All-to-all
  gradients           communication                  communication
```

### Data Parallel (DDP)

**Idea**: Replicate the entire model on each GPU. Split the data batch across GPUs. Each GPU computes gradients on its shard of data, then AllReduce to sync gradients.

- **Pros**: Simple, near-linear scaling for compute-bound models
- **Cons**: Each GPU must hold the **full model** -- impossible for models larger than GPU memory
- **Communication**: AllReduce on gradients (2 * model_size per step)
- **When to use**: Model fits in one GPU, you want faster training

### FSDP (Fully Sharded Data Parallel)

**Idea**: Shard model parameters, gradients, AND optimizer states across GPUs. Before each forward/backward pass for a layer, AllGather the full parameters. After backward, ReduceScatter gradients.

- **Pros**: Memory scales as O(model_size / num_gpus) -- can train models much larger than one GPU
- **Cons**: More communication (AllGather before each layer)
- **Communication**: AllGather parameters + ReduceScatter gradients per layer
- **When to use**: Model doesn't fit in one GPU, want simplicity over pipeline/tensor parallel

### DeepSpeed ZeRO Stages

DeepSpeed's ZeRO (Zero Redundancy Optimizer) progressively shards more state:

| Stage | What's Sharded | Memory per GPU (relative) | Communication |
|-------|---------------|--------------------------|---------------|
| **Stage 0** | Nothing (= DDP) | params + grads + optimizer = ~16x params (fp16+Adam) | AllReduce gradients |
| **Stage 1** | Optimizer states | ~4x reduction in TOTAL training memory (the optimizer states themselves shrink ~N_d-fold, N_d = data-parallel degree) | AllReduce gradients |
| **Stage 2** | Optimizer + gradients | ~8x reduction in total training memory | ReduceScatter grads |
| **Stage 3** | Optimizer + gradients + parameters | ~Nx reduction (N=num GPUs), equivalent to FSDP | AllGather params + ReduceScatter grads |

**Memory breakdown for a 7B param model with fp16 + Adam**:
- Parameters: 7B * 2 bytes = 14 GB
- Gradients: 7B * 2 bytes = 14 GB  
- Optimizer (Adam): 7B * 4 bytes (fp32 copy) + 7B * 4 bytes (momentum) + 7B * 4 bytes (variance) = 84 GB
- Total: ~112 GB on ONE GPU with DDP
- With ZeRO-3 on 8 GPUs: ~14 GB per GPU

### Tensor Parallel (TP)

**Idea**: Split individual layers (weight matrices) across GPUs. For a linear layer `Y = XW`, split W column-wise or row-wise across GPUs.

- **Column parallel**: `Y = [XW1, XW2, ...]` -- split output features
- **Row parallel**: `Y = XW1 + XW2 + ...` -- split input features, AllReduce output
- **Best for**: Intra-node communication (NVLink has ~600 GB/s bandwidth)
- **When to use**: Large individual layers (attention heads, FFN), fast interconnect

### Pipeline Parallel (PP)

**Idea**: Split model layers across GPUs sequentially. GPU 0 gets layers 0-7, GPU 1 gets layers 8-15, etc.

- **Problem**: Naive pipeline has O(num_stages) bubble time (GPUs sit idle)
- **Solution**: Micro-batch scheduling (1F1B = one forward, one backward)
- **1F1B Schedule**: Overlap forward of micro-batch k+1 with backward of micro-batch k
- **Bubble fraction**: (num_stages - 1) / (num_microbatches) -- reduce by increasing micro-batches
- **When to use**: Inter-node communication (slower bandwidth OK since only activations are sent)

### Expert Parallel (EP)

**Idea**: For Mixture of Experts (MoE) models, place different experts on different GPUs. The router decides which tokens go to which expert (and thus which GPU).

- **Communication**: All-to-all to route tokens to correct expert GPUs
- **Load balancing**: Critical -- unbalanced routing wastes GPU compute
- **When to use**: MoE architectures (Mixtral, Switch Transformer)

### Which Parallelism for Which Scenario?

| Scenario | Recommended Strategy |
|----------|---------------------|
| 7B model, 8 GPUs, single node | DDP or FSDP |
| 70B model, 64 GPUs, 8 nodes | TP (intra-node) + PP (inter-node) + FSDP |
| 175B model, 1024 GPUs | TP + PP + FSDP (3D parallelism) |
| MoE model (8x7B) | EP + TP + FSDP |
| Fine-tuning 70B (LoRA) | FSDP (only adapter params are trained) |

In [ ]:
# =============================================================================
# BASIC DDP TRAINING PATTERN (simulated on single GPU)
# =============================================================================
# In real multi-GPU, you'd use torch.distributed.launch or torchrun.
# This shows the PATTERN -- the code structure you'd use.

import torch
import torch.nn as nn

class SimpleTransformerBlock(nn.Module):
    """Simplified transformer block for demonstration."""
    def __init__(self, d_model=512, n_heads=8, d_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Pre-norm architecture (used by most modern LLMs)
        normed = self.norm1(x)
        x = x + self.attn(normed, normed, normed)[0]
        x = x + self.ffn(self.norm2(x))
        return x


class SmallLM(nn.Module):
    """Small language model for demonstrating training patterns."""
    def __init__(self, vocab_size=10000, d_model=512, n_layers=6, n_heads=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(1024, d_model)
        self.layers = nn.ModuleList([
            SimpleTransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)


# ---------- DDP TRAINING PATTERN ----------
# This is what real multi-GPU code looks like.
# We simulate it on one device.

def ddp_training_pattern():
    """
    Real DDP code pattern (simulated on single device).
    
    In production, you would:
    1. torchrun --nproc_per_node=8 train.py
    2. torch.distributed.init_process_group('nccl')
    3. model = DDP(model, device_ids=[local_rank])
    4. sampler = DistributedSampler(dataset)
    """
    # Step 1: Create model
    model = SmallLM(vocab_size=10000, d_model=256, n_layers=4, n_heads=4).to(device)
    
    # In real DDP:
    # model = torch.nn.parallel.DistributedDataParallel(
    #     model, device_ids=[local_rank]
    # )
    
    # Step 2: Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
    
    # Step 3: Training loop
    model.train()
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    print(f"Model memory: {total_params * 4 / 1e6:.1f} MB (fp32)")
    print(f"With Adam optimizer: {total_params * 16 / 1e6:.1f} MB total")
    print()
    
    losses = []
    for step in range(20):
        # Synthetic data
        input_ids = torch.randint(0, 10000, (8, 128), device=device)
        targets = torch.randint(0, 10000, (8, 128), device=device)
        
        # Forward
        logits = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, 10000), targets.view(-1))
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (standard for LLM training)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # In DDP: gradients are automatically AllReduced here
        optimizer.step()
        
        losses.append(loss.item())
        if step % 5 == 0:
            print(f"Step {step:3d} | Loss: {loss.item():.4f}")
    
    return losses

losses = ddp_training_pattern()

plt.figure(figsize=(8, 4))
plt.plot(losses, 'b-o', markersize=3)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('DDP Training Pattern (simulated)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# MEMORY ANALYSIS: Why ZeRO Matters
# =============================================================================

def analyze_training_memory(param_billions: float, num_gpus: int, precision: str = 'fp16'):
    """
    Calculate memory requirements for different parallelism strategies.
    
    For fp16 training with Adam:
    - Parameters: 2 bytes/param (fp16)
    - Gradients: 2 bytes/param (fp16)
    - Optimizer: 4 (fp32 params) + 4 (momentum) + 4 (variance) = 12 bytes/param
    - Total: 16 bytes/param
    """
    params = param_billions * 1e9
    bytes_per_param = 2  # fp16
    
    param_mem = params * bytes_per_param / 1e9  # GB
    grad_mem = params * bytes_per_param / 1e9
    optim_mem = params * 12 / 1e9  # Adam optimizer states
    total_mem = param_mem + grad_mem + optim_mem
    
    results = {
        'DDP (no sharding)': {
            'per_gpu': total_mem,
            'description': 'Full model + grads + optimizer on each GPU'
        },
        'ZeRO Stage 1': {
            'per_gpu': param_mem + grad_mem + optim_mem / num_gpus,
            'description': 'Shard optimizer states only'
        },
        'ZeRO Stage 2': {
            'per_gpu': param_mem + (grad_mem + optim_mem) / num_gpus,
            'description': 'Shard optimizer + gradients'
        },
        'ZeRO Stage 3 / FSDP': {
            'per_gpu': (param_mem + grad_mem + optim_mem) / num_gpus,
            'description': 'Shard everything'
        }
    }
    
    print(f"\n{'='*70}")
    print(f"Memory Analysis: {param_billions}B parameter model, {num_gpus} GPUs")
    print(f"{'='*70}")
    print(f"\nRaw memory breakdown:")
    print(f"  Parameters (fp16):    {param_mem:8.1f} GB")
    print(f"  Gradients (fp16):     {grad_mem:8.1f} GB")
    print(f"  Optimizer (Adam):     {optim_mem:8.1f} GB")
    print(f"  Total per replica:    {total_mem:8.1f} GB")
    print(f"\nPer-GPU memory by strategy:")
    print(f"{'Strategy':<25} {'Per GPU (GB)':>12} {'Fits 80GB?':>10}")
    print(f"{'-'*50}")
    for name, info in results.items():
        fits = 'YES' if info['per_gpu'] < 80 else 'NO'
        marker = ' <--' if fits == 'YES' and name != 'DDP (no sharding)' else ''
        print(f"{name:<25} {info['per_gpu']:>10.1f}  {'  ' + fits}{marker}")
    
    return results


# Analyze different model sizes
for model_size in [7, 13, 70]:
    analyze_training_memory(model_size, num_gpus=8)

In [ ]:
# =============================================================================
# VISUALIZATION: Parallelism Strategies
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Plot 1: Memory per GPU vs Model Size ---
model_sizes = [1, 3, 7, 13, 30, 70, 175]
num_gpus = 8

strategies = {
    'DDP': [],
    'ZeRO-1': [],
    'ZeRO-2': [],
    'ZeRO-3/FSDP': []
}

for size in model_sizes:
    params = size * 1e9
    p = params * 2 / 1e9
    g = params * 2 / 1e9
    o = params * 12 / 1e9
    strategies['DDP'].append(p + g + o)
    strategies['ZeRO-1'].append(p + g + o / num_gpus)
    strategies['ZeRO-2'].append(p + (g + o) / num_gpus)
    strategies['ZeRO-3/FSDP'].append((p + g + o) / num_gpus)

colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
for (name, mems), color in zip(strategies.items(), colors):
    axes[0].plot(model_sizes, mems, 'o-', label=name, color=color, linewidth=2)

axes[0].axhline(y=80, color='gray', linestyle='--', label='A100 80GB limit')
axes[0].set_xlabel('Model Size (B params)')
axes[0].set_ylabel('Memory per GPU (GB)')
axes[0].set_title('Memory per GPU vs Model Size (8 GPUs)')
axes[0].legend(fontsize=8)
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Communication Overhead ---
categories = ['DDP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3\n/FSDP', 'Tensor\nParallel', 'Pipeline\nParallel']
comm_volume = [1.0, 1.0, 1.0, 1.5, 2.0, 0.3]  # Relative to DDP
comm_freq = [1, 1, 1, 3, 6, 1]  # Operations per step (relative)

bar_colors = ['#e74c3c', '#e67e22', '#f39c12', '#3498db', '#9b59b6', '#1abc9c']
bars = axes[1].bar(categories, comm_volume, color=bar_colors, alpha=0.8)
axes[1].set_ylabel('Relative Communication Volume')
axes[1].set_title('Communication Overhead by Strategy')
axes[1].grid(True, alpha=0.3, axis='y')

# --- Plot 3: Pipeline Parallel Bubble ---
stages = 4
microbatches_list = [4, 8, 16, 32]

for mb in microbatches_list:
    # Utilization = m / (m + p - 1): m micro-batches of work over a schedule
    # spanning m + p - 1 slots (e.g., p=4, m=4 -> 57%).
    # (The naive 1 - (p-1)/m formula is WRONG: it understates utilization and
    # goes negative for m < p - 1.)
    efficiency = mb / (mb + stages - 1)
    axes[2].bar(f'MB={mb}', efficiency * 100, 
                color=plt.cm.viridis(efficiency), alpha=0.8)
    axes[2].text(microbatches_list.index(mb), efficiency * 100 + 1, 
                 f'{efficiency*100:.0f}%', ha='center', fontsize=10)

axes[2].set_ylabel('GPU Utilization (%)')
axes[2].set_title(f'Pipeline Parallel Efficiency ({stages} stages)')
axes[2].set_ylim(0, 110)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 5. Inference Optimization

Training a model is half the battle. Serving it at scale -- with low latency and high throughput -- is where the real systems engineering happens.

### 5.1 KV-Cache: The Foundational Optimization

In autoregressive generation, each new token attends to ALL previous tokens. Without caching, this means recomputing all key-value pairs at each step -- O(n^2) total compute for n tokens.

**KV-Cache stores past key/value pairs** so each new token only needs to compute its own K,V and attend to the cached history. Careful: this avoids only the recomputation of K/V for past tokens -- attention for each new token still reads the whole cache, so per-token attention cost remains O(n), not O(1).

**But**: KV-cache memory grows linearly with sequence length AND batch size:
- For Llama-2 70B (80 layers, GQA with 8 KV heads, head_dim 128) in fp16 with seq_len=4096, batch_size=1: KV-cache = 2 x 80 x 8 x 128 x 4096 x 2 bytes ~ **1.34 GB**
- For batch_size=32: ~43 GB (!!)
- **Classic interview trap**: forgetting that Llama-2-70B uses GQA (8 KV heads, not 64) inflates the estimate ~8x (to ~10.7 GB per 4K sequence)

In [ ]:
# =============================================================================
# KV-CACHE IMPLEMENTATION: See the speedup yourself
# =============================================================================

class SimpleAttention(nn.Module):
    """Attention with optional KV-cache."""
    def __init__(self, d_model=256, n_heads=8):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, kv_cache=None, use_cache=False):
        B, T, _ = x.shape
        
        # Compute Q, K, V for current input
        q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        # If we have cached K, V, concatenate
        if kv_cache is not None:
            cached_k, cached_v = kv_cache
            k = torch.cat([cached_k, k], dim=2)  # Append new K to cache
            v = torch.cat([cached_v, v], dim=2)  # Append new V to cache
        
        # Standard attention
        scale = self.d_head ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.W_o(out)
        
        new_cache = (k, v) if use_cache else None
        return out, new_cache


# Benchmark: with vs without KV-cache
attn = SimpleAttention(d_model=256, n_heads=8).to(device)

seq_lengths = [32, 64, 128, 256, 512]
times_no_cache = []
times_with_cache = []

for seq_len in seq_lengths:
    x_full = torch.randn(1, seq_len, 256, device=device)
    
    # WITHOUT cache: recompute everything for each new token
    torch.cuda.synchronize() if device.type == 'cuda' else None
    start = time.perf_counter()
    for t in range(10):
        # Simulate: at each step, process all tokens up to position t
        _ = attn(x_full[:, :seq_len], use_cache=False)
    torch.cuda.synchronize() if device.type == 'cuda' else None
    times_no_cache.append((time.perf_counter() - start) / 10)
    
    # WITH cache: only process new token, use cached K, V
    torch.cuda.synchronize() if device.type == 'cuda' else None
    start = time.perf_counter()
    for t in range(10):
        cache = None
        for pos in range(0, seq_len, 32):  # Process in chunks of 32
            chunk = x_full[:, pos:pos+32]
            _, cache = attn(chunk, kv_cache=cache, use_cache=True)
    torch.cuda.synchronize() if device.type == 'cuda' else None
    times_with_cache.append((time.perf_counter() - start) / 10)

# NOTE on this benchmark's limitations:
# 1) The "without cache" branch runs ONE full-sequence forward (a single
#    prefill), not true no-cache decoding, which would re-encode the whole
#    prefix for EVERY new token. So this UNDERESTIMATES the real gap.
# 2) This toy attention has no causal mask (every position attends to all
#    positions) -- fine for a timing demo, not a correct causal LM.
# In real inference, the cache advantage is even larger because you process
# ONE token at a time during generation.

plt.figure(figsize=(10, 5))
plt.plot(seq_lengths, [t*1000 for t in times_no_cache], 'r-o', label='Without KV-Cache', linewidth=2)
plt.plot(seq_lengths, [t*1000 for t in times_with_cache], 'g-o', label='With KV-Cache (chunked)', linewidth=2)
plt.xlabel('Sequence Length')
plt.ylabel('Time (ms)')
plt.title('KV-Cache Speedup')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSpeedups:")
for sl, t_no, t_yes in zip(seq_lengths, times_no_cache, times_with_cache):
    print(f"  Seq len {sl:4d}: {t_no/t_yes:.2f}x faster with cache")

In [ ]:
# =============================================================================
# KV-CACHE MEMORY CALCULATOR
# =============================================================================

def kv_cache_memory(n_layers, d_model, n_heads, seq_len, batch_size,
                    n_kv_heads=None, dtype_bytes=2):
    """
    Calculate KV-cache memory.

    KV-cache stores K and V for each layer, for each KV head.
    With grouped-query attention (GQA), n_kv_heads < n_heads, shrinking the
    cache by n_heads / n_kv_heads. Defaults to MHA (n_kv_heads = n_heads).

    Per layer: 2 (K+V) * batch_size * n_kv_heads * seq_len * d_head * dtype_bytes
    Total: n_layers * above
    """
    if n_kv_heads is None:
        n_kv_heads = n_heads  # MHA default
    d_head = d_model // n_heads
    per_layer = 2 * batch_size * n_kv_heads * seq_len * d_head * dtype_bytes
    total = n_layers * per_layer
    return total / 1e9  # GB


# Real model configurations -- n_kv_heads matters! Llama-2 70B uses GQA (8 KV heads).
models = {
    'Llama-2 7B':  {'n_layers': 32, 'd_model': 4096, 'n_heads': 32, 'n_kv_heads': 32},   # MHA
    'Llama-2 13B': {'n_layers': 40, 'd_model': 5120, 'n_heads': 40, 'n_kv_heads': 40},   # MHA
    'Llama-2 70B': {'n_layers': 80, 'd_model': 8192, 'n_heads': 64, 'n_kv_heads': 8},    # GQA!
    'GPT-4 scale':  {'n_layers': 120, 'd_model': 12288, 'n_heads': 96, 'n_kv_heads': 96},  # hypothetical, assume MHA
}

print(f"{'Model':<15} {'Seq Len':>8} {'Batch':>6} {'KV-Cache (GB)':>14} {'Notes':>20}")
print('=' * 70)

for name, cfg in models.items():
    for seq_len, batch_size in [(4096, 1), (4096, 32), (32768, 1), (128000, 1)]:
        mem = kv_cache_memory(cfg['n_layers'], cfg['d_model'], cfg['n_heads'],
                              seq_len, batch_size, n_kv_heads=cfg['n_kv_heads'])
        note = ''
        if mem > 80:
            note = 'EXCEEDS A100 80GB'
        elif mem > 40:
            note = 'Tight on A100'
        print(f"{name:<15} {seq_len:>8,} {batch_size:>6} {mem:>13.1f}  {note}")
    print()

# The classic interview trap: forgetting GQA.
mha_wrong = kv_cache_memory(80, 8192, 64, 4096, 1)                 # wrongly assumes 64 KV heads
gqa_right = kv_cache_memory(80, 8192, 64, 4096, 1, n_kv_heads=8)   # Llama-2 70B reality
print(f"Llama-2 70B @ 4K ctx, batch 1 -- assuming MHA: {mha_wrong:.2f} GB; with GQA (8 KV heads): {gqa_right:.2f} GB")
print(f"Forgetting GQA inflates the KV-cache estimate ~{mha_wrong / gqa_right:.0f}x. Classic interview trap!")


### 5.2 PagedAttention (vLLM)

**Paper**: [Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180)

**The Problem**: KV-cache is allocated contiguously per request. When requests have variable lengths, this leads to:
- **Internal fragmentation**: Pre-allocated cache is larger than needed
- **External fragmentation**: Gaps between deallocated caches can't be reused
- Result: Up to 60-80% of KV-cache memory is wasted

**The Insight**: Treat KV-cache like OS virtual memory. Break it into fixed-size **pages** (blocks of tokens). Map logical token positions to physical memory blocks via a **page table**.

**How it works**:
1. KV-cache is divided into fixed-size blocks (e.g., 16 tokens per block)
2. Each request gets a page table mapping logical blocks to physical blocks
3. Physical blocks are allocated on demand (no pre-allocation waste)
4. When a request finishes, its physical blocks return to a free pool
5. Blocks can be shared (copy-on-write) for beam search, parallel sampling

**Impact**: 
- Near-zero memory waste (< 4% internal fragmentation from last block)
- Up to 24x throughput vs the naive HuggingFace Transformers baseline; 2-4x vs the serious serving systems of the time (Orca / FasterTransformer / TGI-class)
- Enables larger batch sizes = higher GPU utilization

### 5.3 Continuous Batching

**Traditional (static) batching**: Wait until batch is full, process all sequences until ALL finish. Short sequences waste compute waiting for long ones.

**Continuous batching**: As soon as any sequence finishes, immediately add a new request to the batch. No GPU cycles wasted on padding.

```
Static Batching:         Continuous Batching:
                         
Req1: ████████           Req1: ████████
Req2: ████░░░░           Req2: ████
Req3: ██░░░░░░           Req3: ██
                              Req4: ██████
     ^ wasted time                Req5: ████
                              ^ no waste: new reqs fill gaps
```

### 5.4 Speculative Decoding

**Idea**: Use a small, fast "draft" model to generate N candidate tokens. Then verify all N tokens with the large model **in a single forward pass** (because verification is parallelizable -- just run the large model on all N tokens at once).

**Why it works**:
- Autoregressive generation is **memory-bandwidth bound**, not compute-bound
- The large model's forward pass costs nearly the same for 1 token or N tokens (batch dimension is free when memory-bound)
- If the draft model agrees with the large model on K out of N tokens, you get K+1 tokens for the cost of ~1 large model forward pass

**The acceptance rule (what makes it exact)**: each draft token x is accepted with probability min(1, p_target(x) / p_draft(x)); on rejection, the token is resampled from the adjusted distribution norm(max(0, p_target - p_draft)) and the round stops. This rejection-sampling scheme is what guarantees the output distribution is exactly the target model's. If all k draft tokens are accepted, the verification pass also yields one bonus (k+1)-th token from the target model for free

**EAGLE-3** (state of the art):
- 3-6x speedup with **no quality loss** (provably identical output distribution)
- Uses a lightweight head on top of the target model's hidden states
- Tree-based verification: explore multiple continuation paths simultaneously

**Key constraint**: Only works when draft model acceptance rate is high. For creative/diverse generation, acceptance rate drops.

### 5.5 Quantization

| Method | Approach | Precision | Pros | Cons |
|--------|----------|-----------|------|------|
| **GPTQ** | Post-training, weight-only | INT4/INT8 | Fast inference | Requires calibration data |
| **AWQ** | Activation-aware weight quant | INT4 | Better quality than GPTQ | Slightly slower |
| **GGUF** | CPU-friendly format | Various | Runs on CPU/laptop | Slower than GPU |
| **FP8** | Native on H100 | FP8 | No quality loss, 2x speedup | Requires H100 |
| **SmoothQuant** | Smooth activations, quant both W and A | INT8 | Good for W8A8 | Requires calibration |

**Rule of thumb**: INT4 weight-only quantization typically loses < 1% on benchmarks for models > 7B.

### 5.6 Prefix Caching

**Idea**: Many requests share common prefixes (system prompts, few-shot examples). Cache the KV pairs for these shared prefixes and reuse them.

- System prompt (500 tokens) is the same for all requests -> compute KV once, reuse for all
- Few-shot examples -> cache and reuse
- vLLM supports automatic prefix caching via hash-based deduplication
- SGLang's RadixAttention: tree-based prefix caching for even more sharing

In [ ]:
# =============================================================================
# SPECULATIVE DECODING: Simplified Implementation
# =============================================================================

def simulate_speculative_decoding(
    num_tokens: int = 100,
    draft_acceptance_rate: float = 0.7,
    lookahead: int = 5,
    large_model_time_ms: float = 50.0,
    draft_model_time_ms: float = 5.0,
):
    """
    Simulate speculative decoding to show the speedup.
    
    Standard autoregressive: 1 large model call per token
    Speculative: N draft calls + 1 large model verification call per attempt

    NOTE: real speculative sampling accepts each draft token with probability
    min(1, p_target/p_draft) and, on rejection, resamples from the adjusted
    distribution norm(max(0, p_target - p_draft)) -- that rule is what makes
    the output distribution exactly the target model's. It also grants a
    BONUS (k+1)-th token when all k drafts are accepted; this toy simulation
    omits that bonus token, so it slightly understates the speedup.
    """
    np.random.seed(42)
    
    # Standard autoregressive
    standard_time = num_tokens * large_model_time_ms
    
    # Speculative decoding
    speculative_time = 0
    tokens_generated = 0
    num_attempts = 0
    accepted_per_attempt = []
    
    while tokens_generated < num_tokens:
        num_attempts += 1
        
        # Draft model generates N tokens
        speculative_time += lookahead * draft_model_time_ms
        
        # Large model verifies all N tokens in one pass
        speculative_time += large_model_time_ms
        
        # How many tokens are accepted?
        accepted = 0
        for i in range(lookahead):
            if np.random.random() < draft_acceptance_rate:
                accepted += 1
            else:
                accepted += 1  # We always get at least up to the first rejection + correct token
                break
        
        tokens_generated += accepted
        accepted_per_attempt.append(accepted)
    
    speedup = standard_time / speculative_time
    avg_accepted = np.mean(accepted_per_attempt)
    
    return {
        'standard_time_ms': standard_time,
        'speculative_time_ms': speculative_time,
        'speedup': speedup,
        'num_attempts': num_attempts,
        'avg_accepted_per_attempt': avg_accepted,
        'tokens_generated': tokens_generated
    }


# Sweep over acceptance rates
acceptance_rates = np.arange(0.3, 0.96, 0.05)
lookaheads = [3, 5, 8]

plt.figure(figsize=(10, 6))
for la in lookaheads:
    speedups = []
    for ar in acceptance_rates:
        result = simulate_speculative_decoding(
            num_tokens=200,
            draft_acceptance_rate=ar,
            lookahead=la
        )
        speedups.append(result['speedup'])
    plt.plot(acceptance_rates * 100, speedups, 'o-', label=f'Lookahead={la}', linewidth=2)

plt.xlabel('Draft Model Acceptance Rate (%)')
plt.ylabel('Speedup vs Standard Autoregressive')
plt.title('Speculative Decoding Speedup')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Show specific example
result = simulate_speculative_decoding(num_tokens=200, draft_acceptance_rate=0.75, lookahead=5)
print(f"\nExample (75% acceptance rate, lookahead=5):")
print(f"  Standard time: {result['standard_time_ms']:.0f} ms")
print(f"  Speculative time: {result['speculative_time_ms']:.0f} ms")
print(f"  Speedup: {result['speedup']:.2f}x")
print(f"  Avg tokens accepted per attempt: {result['avg_accepted_per_attempt']:.1f}")

In [ ]:
# =============================================================================
# INFERENCE OPTIMIZATION SUMMARY VISUALIZATION
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

optimizations = [
    'KV-Cache',
    'Continuous\nBatching', 
    'PagedAttention',
    'Speculative\nDecoding',
    'INT4\nQuantization',
    'Prefix\nCaching',
    'FlashAttention',
    'All Combined'
]

# Approximate throughput multipliers (tokens/sec improvement)
throughput_gain = [10, 3, 2.5, 3.5, 2, 1.5, 2, 100]
# Whether they compose multiplicatively or not
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c', '#f39c12', '#2c3e50']

bars = ax.bar(optimizations, throughput_gain, color=colors, alpha=0.85)

for bar, gain in zip(bars, throughput_gain):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{gain}x', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('Throughput Multiplier (approx.)')
ax.set_title('Inference Optimization Impact (approximate, varies by workload)')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 6. Practice Problem 1: Distributed RLHF Training Pipeline for 70B Model

### Problem Statement

> **"Design a distributed RLHF training pipeline for a 70B parameter model."**

### Clarifying Questions to Ask

- What's the available GPU cluster? (e.g., 512x A100 80GB)
- What RLHF algorithm? (PPO vs DPO vs GRPO)
- What's the sequence length? (2K? 8K? 32K?)
- What's the target throughput? (samples/hour)
- Online (with generation) or offline (pre-collected)?  

### Solution Outline

#### A) The 4 Models in RLHF (PPO)

| Model | Size | Trainable? | Purpose |
|-------|------|------------|----------|
| **Policy** | 70B | YES | The model being trained |
| **Reference** | 70B | NO (frozen) | KL penalty anchor |
| **Reward Model** | 7-13B | NO (frozen) | Scores outputs |
| **Value Model** | 7-13B | YES | Estimates advantages |

Total memory: 70B (policy) + 70B (reference) + 13B (reward) + 13B (value) = ~166B params to hold in memory simultaneously.

#### B) Memory Constraints & Parallelism Strategy

- **Policy (70B, trainable)**: FSDP/ZeRO-3 across 64 GPUs. With fp16 + Adam: 70B * 16 bytes = 1.12 TB -> 17.5 GB/GPU across 64 GPUs. Plus activations and KV-cache.
- **Reference (70B, frozen)**: Tensor parallel across 8 GPUs (single node). Only need inference, no optimizer states.
- **Reward Model (13B, frozen)**: Tensor parallel across 4 GPUs.
- **Value Model (13B, trainable)**: FSDP across 8 GPUs.

#### C) The Generation Bottleneck

The critical bottleneck in online RLHF is **generation**: the policy model must generate responses autoregressively before training on them.

Solutions:
1. **Separate generation and training clusters**: Dedicate some GPUs to generation (with inference optimizations like vLLM), others to training.
2. **Asynchronous generation**: Pipeline generation and training. While batch N is training, batch N+1 is being generated.
3. **Use quantized model for generation**: INT8 policy for generation, fp16 for training.

#### D) Pipeline

```
Prompts -> [Generation (Policy)] -> Responses
                                        |
                                        v
                              [Reward Model] -> Rewards
                              [Reference Model] -> KL penalty
                                        |
                                        v
                              [Advantage Estimation (Value + GAE)]
                                        |
                                        v
                              [PPO Update (Policy + Value)]
                                        |
                                        v
                              [Checkpoint + Eval] -> repeat
```

#### E) Key Design Decisions

- **Async vs sync generation**: Async is faster but the policy may drift between generation and training. Use a staleness threshold.
- **Checkpointing**: Every N PPO steps. Use async checkpointing (don't block training).
- **Evaluation gates**: Run eval suite every K steps. If quality drops, rollback.
- **KL coefficient**: Start high (0.2), anneal down. Monitor KL divergence.
- **Reward hacking**: Monitor reward vs human eval correlation. If they diverge, reward model may be exploited.

---
## 7. Practice Problem 2: Serving System for 10K QPS LLM Inference

### Problem Statement

> **"Design a serving system for 10,000 queries per second LLM inference."**

### Clarifying Questions

- What model size? (7B? 70B?)
- What's the average input/output length?
- What are the latency SLOs? (p50, p99, p999)
- Is this streaming (SSE) or batch responses?
- Geographic distribution of users?

### Solution Outline

**Assumptions**: 70B model, avg 500 input / 200 output tokens, p99 < 2s TTFT, p50 < 100ms inter-token latency.

#### A) Back-of-envelope Capacity (do this honestly -- the naive number is huge)

- Effective decode throughput for a 70B is on the order of ~150 output tok/s per GPU (varies with batching, quantization, hardware)
- At our assumed ~200-token responses, one GPU sustains only ~0.75 responses/s (150 / 200); with ~300-token responses it drops to ~0.5 responses/s
- Naive requirement for 10K QPS: 10,000 / 0.75 ~ **13,000+ GPUs** (~20,000 at 300-token responses). Beware folklore like "50-100 req/s per A100 for a 70B" -- that is off by orders of magnitude
- The design levers are what close the gap:
  - **Aggressive continuous batching**: many concurrent decode streams per GPU raise aggregate tok/s several-fold
  - **Shorter outputs**: response-length limits / concise modes cut per-request cost linearly
  - **Prefix caching**: shared system prompts save 30-70% of prefill compute
  - **Model tiering / routing**: send the majority of traffic to much smaller models (7-13B class); reserve the 70B for hard queries
  - **Quantization** (INT4/FP8): more throughput per GPU
- After these levers, a realistic 70B-fronted deployment lands in the **hundreds-to-thousands of GPUs**, depending on how much traffic the smaller tiers absorb. In the interview, walk through this reduction explicitly instead of quoting a small GPU count up front

#### B) Architecture

```
Clients -> CDN/Edge -> Load Balancer (L7)
                            |
                    [Request Router]
                    /       |       \
             [vLLM Pool 1] [Pool 2] [Pool N]
              (8xA100)    (8xA100)  (8xA100)
                    \       |       /
                    [Response Cache]
                            |
                    [Metrics + Logging]
```

#### C) Key Components

1. **Load Balancing**: 
   - Least-connections or least-queue-depth (not round-robin -- requests have variable compute cost)
   - Consider input length in routing: short requests to less-loaded servers

2. **Batching Strategy**:
   - Continuous batching (vLLM/TGI style)
   - Max batch size limited by GPU memory (KV-cache budget)
   - Priority queues: premium users get lower latency

3. **KV-Cache Management**:
   - PagedAttention for memory efficiency
   - Prefix caching for system prompts
   - Eviction policy: LRU for prefix cache, immediate free for completed requests

4. **Autoscaling**:
   - Scale on queue depth, not CPU utilization
   - Pre-warm instances (model loading takes minutes)
   - Predictive scaling based on traffic patterns

5. **SLOs**:
   - TTFT (time to first token): p50 < 500ms, p99 < 2s
   - ITL (inter-token latency): p50 < 50ms, p99 < 200ms
   - Throughput: 10K QPS sustained
   - Error rate: < 0.1%

6. **Cost Optimization**:
   - Spot/preemptible instances for non-latency-sensitive traffic
   - Request caching (semantic dedup for identical prompts)
   - Dynamic quantization: INT4 during peak, FP16 during off-peak for quality
   - Approximate cost: scales with the fleet from (A); e.g., 2,000 A100s * $2/hr = $4,000/hr = ~$35M/year -- which is why model tiering and output-length control dominate the cost conversation

---
## 8. Practice Problem 3: Agent Platform for 1M Concurrent Sessions

### Problem Statement

> **"Design an agent platform supporting 1 million concurrent sessions."**

### Clarifying Questions

- What kinds of tools can agents use? (web search, code execution, file access)
- What's the average session duration? (minutes? hours? days?)
- What's the average number of LLM calls per session?
- Multi-turn or single-turn?
- What safety/sandboxing requirements?

### Solution Outline

#### A) Scale Analysis

- 1M concurrent sessions
- Assume avg 5-min session with 10 LLM calls
- Peak LLM call rate: 1M * 10 / 300s = ~33K QPS to LLM
- Not all sessions are actively calling LLM at once. Assume 10% active at any moment = ~3.3K QPS

#### B) Architecture

```
                        [API Gateway]
                             |
                    [Session Manager]
                    /       |        \
           [Orchestrator] [Orchestrator] [Orchestrator]
            (stateless)   (stateless)    (stateless)
                |            |              |
         [LLM Service]  [Tool Service]  [Memory Service]
                             |
                    [Sandbox Pool]
                    (gVisor/Firecracker)
```

#### C) Key Design Decisions

1. **Session Management**:
   - Sessions are stored in Redis (hot) + S3 (cold)
   - Session state: conversation history, tool outputs, agent state machine
   - Evict idle sessions to cold storage after 5 min inactivity
   - Resume from cold storage on reactivation (~500ms penalty)

2. **Tool Execution Sandboxing**:
   - Code execution in Firecracker micro-VMs (boot in <125ms)
   - Network-isolated: allowlist for approved APIs
   - Resource limits: 2 vCPU, 512MB RAM, 30s timeout per tool call
   - Pool of pre-warmed sandboxes (reduce cold start)

3. **Context Window Management**:
   - Sliding window with summarization: keep last N turns verbatim, summarize older turns
   - Retrieval-augmented: store all turns in vector DB, retrieve relevant ones
   - Budget: track token usage per session, enforce limits

4. **Cost Control Per Session**:
   - Token budget per session (e.g., 100K tokens max)
   - Tool call budget (e.g., 50 tool calls max)
   - Compute budget (e.g., 60s total sandbox time)
   - Rate limiting: max 10 LLM calls/minute per session

5. **Reliability**:
   - Orchestrator is stateless -> any instance can resume any session
   - Idempotent tool calls where possible
   - Retry with exponential backoff for transient failures
   - Circuit breaker for tool services
   - Dead letter queue for failed sessions

---
## 9. Practice Problem 4: Reward Model Serving with A/B Testing

### Problem Statement

> **"Design a reward model serving system with A/B testing capabilities."**

### Clarifying Questions

- How many reward models do we want to serve simultaneously?
- What's the QPS for reward scoring?
- Online (real-time scoring) or offline (batch evaluation)?
- What metrics define "better" for a reward model?
- How quickly do we need statistical significance?

### Solution Outline

#### A) Architecture

```
[Policy Model] generates responses
        |
        v
[Traffic Splitter] ---> 90% [RM v2 (production)]
        |                    -> scores used for ranking/RLHF
        +----------> 10% [RM v3 (candidate)]
                         -> scores logged but not used
        |
        v
[Metric Collector] -> [Analysis Pipeline]
                            |
                    [Statistical Significance Engine]
                            |
                    [Auto-promote or rollback]
```

#### B) Model Versioning

- Model registry: version, training data hash, hyperparams, eval metrics
- Blue-green deployment: candidate model deployed alongside production
- Shadow mode: candidate scores everything but results aren't used
- Canary mode: candidate scores are used for a small traffic slice

#### C) Traffic Splitting

- Hash-based splitting on user_id for consistency (same user always sees same model)
- Configurable split ratio (start at 1%, ramp to 5%, 10%, 50%, 100%)
- Kill switch: instantly route 100% to production if issues detected

#### D) Metric Collection

**Primary metrics**:
- Agreement with human preferences (gold-labeled data)
- Correlation with downstream task performance
- Calibration (are scores well-distributed?)
- Latency (does the new model serve faster/slower?)

**Guardrail metrics**:
- Reward hacking indicators (scores drifting higher without quality improvement)
- Distribution shift detection (score histograms shifting)
- Error rate

#### E) Statistical Significance

- Sequential testing (not fixed-horizon) to enable early stopping
- Use CUPED (Controlled-experiment Using Pre-Experiment Data) for variance reduction
- Minimum detectable effect: 0.5% agreement improvement
- Significance level: alpha = 0.05, power = 0.8
- Expected time to significance: ~1-2 weeks at 10% traffic

#### F) Rollback

- Automated rollback if:
  - Error rate > 2x production
  - Latency p99 > 1.5x production
  - Guardrail metric triggers
- Manual rollback with one-click
- Post-mortem analysis pipeline

---
## 10. Practice Problem 5: Post-Training Pipeline for a Frontier LLM

### Problem Statement

> **"Design the post-training pipeline for a frontier LLM."**

### Clarifying Questions

- What's the base model size? (70B? 400B?)
- What capabilities do we want? (instruction following, coding, reasoning, safety)
- What's the budget? (GPU hours, human annotation budget)
- What's the iteration cycle time target? (days? weeks?)
- What safety requirements?

### Solution Outline

#### A) Pipeline Overview

```
[Base Model (pre-trained)]
        |
        v
[SFT Stage 1: General instruction following]
        |  Data: ~100K high-quality instruction-response pairs
        |  Eval: MT-Bench, AlpacaEval
        v
[SFT Stage 2: Specialized capabilities]
        |  Data: code (50K), math (30K), reasoning (20K)
        |  Eval: HumanEval, GSM8K, MATH
        v
[Reward Model Training]
        |  Data: 100K+ comparison pairs from human annotators
        |  Eval: agreement rate, calibration
        v
[RLHF/DPO/GRPO]
        |  Iterate until eval metrics plateau
        v
[Safety Training (RLHF with safety RM)]
        |  Red-teaming -> collect violations -> train
        v
[Evaluation Gate]
        |  Must pass ALL benchmarks + safety evals
        v
[Deployment]
```

#### B) Data Pipeline

1. **Collection**:
   - Human-written demonstrations (expensive, high quality)
   - Model-generated + human-filtered (cheaper, scalable)
   - Synthetic data from stronger models (distillation)
   - Mining from public datasets (careful dedup and filtering)

2. **Annotation**:
   - Comparison annotations for reward model (A vs B, with rubric)
   - Quality rubric: helpfulness, harmlessness, honesty
   - Inter-annotator agreement monitoring (drop annotators below threshold)
   - Active learning: select most informative examples for annotation

3. **Quality Control**:
   - Deduplication (exact + near-duplicate with MinHash)
   - Toxicity filtering
   - PII removal
   - Length filtering (remove too short/long)
   - Difficulty stratification (ensure coverage of easy/medium/hard)

#### C) Training Strategy

- **SFT**: Learning rate 1e-5, cosine schedule, 2-3 epochs
- **RM**: Learning rate 1e-5, trained to convergence on comparison data
- **PPO**: KL coeff 0.2, clip ratio 0.2, 4 PPO epochs per batch
- **DPO** (alternative): beta=0.1, simpler than PPO but less flexible
- **GRPO** (alternative): Group Relative Policy Optimization, no value model needed

#### D) Evaluation Gates

Each stage must pass before proceeding:

| Stage | Must Pass | Threshold |
|-------|-----------|----------|
| Post-SFT | MT-Bench, AlpacaEval | > previous best |
| Post-RM | Agreement with held-out human prefs | > 70% |
| Post-RLHF | All benchmarks + safety evals | > previous best, 0 safety failures |
| Pre-deploy | Red team evaluation | 0 critical findings |

#### E) Iteration Cycles

- **Fast loop** (daily): Analyze eval results, adjust hyperparameters, re-run
- **Medium loop** (weekly): New data batches, retrain RM, new RLHF run
- **Slow loop** (monthly): New data collection campaigns, annotation guidelines updates, architecture changes

#### F) Human Feedback Loop

```
Deploy -> Users interact -> Sample conversations -> Human review
    -> Identify failure modes -> Create training data -> Retrain
    -> Deploy improved model -> Repeat
```

Key: The cycle time of this loop determines how fast the model improves.

---
## 11. "Why Does This Work?" -- Interview Meta-Strategy

### Common Mistakes in System Design Interviews

| Mistake | Why It's Bad | Fix |
|---------|-------------|-----|
| **Jumping to solution** | Shows you don't think about requirements | Always spend 5 min on requirements + clarifying questions |
| **Not clarifying requirements** | You'll design the wrong system | Ask: "What's the scale? What are the SLOs? What's the cost budget?" |
| **Ignoring cost** | Frontier labs care deeply about GPU costs | Always do back-of-envelope cost analysis |
| **Overcomplicating** | Shows poor judgment about trade-offs | Start simple, add complexity only when justified |
| **No numbers** | Hand-wavy design is unconvincing | "How many GPUs?" should be answerable |
| **Single-point-of-failure** | Shows lack of production experience | Always ask: "What happens when X fails?" |
| **Ignoring data quality** | Garbage in, garbage out | Data pipeline is often the most important component |

### What Interviewers Look For

1. **Structured thinking**: Can you decompose a vague problem into concrete components?
2. **Trade-off awareness**: Can you articulate why you chose X over Y?
3. **Scalability reasoning**: Can you reason about what breaks at 10x scale?
4. **Depth on demand**: Can you go deep when probed on a specific component?
5. **Practical experience signals**: Have you actually built and operated these systems?
6. **Cost awareness**: Can you estimate GPU hours, storage costs, bandwidth?

### The Golden Rule

> **"Start with the simplest thing that could work, then add complexity as requirements demand it."**

Don't start with a multi-region, multi-model, auto-scaling, A/B-testing system. Start with "one GPU, one model, one user." Then scale up.

---
## Interview Gauntlet: ML System Design (THE Critical Round)

*System design is often the make-or-break round for senior/principal roles. You get 45 minutes. The interviewer is looking for structured thinking, real numbers, and the ability to make and justify tradeoffs. This section gives you the actual framework and numbers you need.*

---

### Numbers You Must Have Memorized

Before we get to questions, burn these into your brain:

| Resource | Spec | Why It Matters |
|----------|------|----------------|
| H100 GPU | 80GB HBM3, 3.35 TB/s bandwidth | The unit of all calculations |
| H100 cost | ~$30K purchase / ~$3/hr cloud | For cost estimates |
| 70B model fp16 | ~140GB (2 bytes x 70B params) | Minimum 2 H100s |
| 70B model INT4 | ~35GB (0.5 bytes x 70B params) | Fits on 1 H100 |
| Inference throughput | 50-200 tok/s per H100 for 70B | Batch-size dependent |
| Training throughput | ~3000 tokens/sec/GPU for pretraining | For timeline estimates |
| Llama 3 405B training | 16K H100s, ~54 days | Reference point for large runs |
| InfiniBand bandwidth | 400 Gbps per link | For communication bottleneck analysis |
| GPU failure rate | MTBF ~30 min for 10K GPU cluster | Why checkpointing matters |
| Checkpoint time | 5-15 min for a large model | Determines checkpoint frequency |

### The 45-Minute Framework

| Phase | Time | What to Do |
|-------|------|------------|
| Requirements | 5 min | Clarify scope: model size, latency SLO, QPS, budget, data constraints |
| High-level architecture | 5 min | Draw the system diagram, name every component |
| Deep design | 25 min | Design 2-3 key components in detail with numbers |
| Monitoring & cost | 5 min | How you know it's working, what it costs |
| Q&A buffer | 5 min | Interviewer deep-dives into one component |

### Question 1 (45 min): "Design the training infrastructure for a 400B MoE model."

**Clarify requirements first** (show the interviewer you think before you design):
- Model: 400B total params, 8 experts, ~50B active per token
- Data: 15T tokens target
- Timeline: 3 months
- Budget: what's the constraint?

**Hardware selection:**
- 4096 H100 GPUs (512 nodes x 8 GPUs/node)
- InfiniBand interconnect: 400 Gbps per GPU, fat-tree topology
- Shared storage: Lustre or GPFS, high-throughput for data loading
- Cost estimate: 4096 GPUs x $3/hr x 24hr x 90 days = ~$26.5M compute alone. Realistically $50-100M including overhead, failed runs, storage, networking.

**Parallelism strategy (3D parallelism + expert parallelism):**
- **Tensor Parallelism (TP=8):** Within each node (8 GPUs connected by NVLink at 900 GB/s). Split attention heads and FFN columns across GPUs.
- **Expert Parallelism (EP):** Route experts across nodes. Each node holds a subset of experts. Token routing via top-k gating.
- **Pipeline Parallelism (PP=8):** Across nodes. 8-stage pipeline with micro-batching to keep GPUs busy.
- **Data Parallelism (DP=64):** Across groups of nodes. Each DP replica processes different data. Gradient all-reduce across replicas.
- Total: TP=8 x PP=8 x DP=64 = 4096 GPUs. Check.

**MoE-specific considerations:**
- Expert load balancing: auxiliary loss to prevent expert collapse (all tokens going to 1-2 experts). Modern alternative: DeepSeek-V3's auxiliary-loss-FREE balancing, which adjusts per-expert bias terms on the routing scores instead of adding an aux loss
- All-to-all communication for expert routing: this is the bottleneck. Design the routing to minimize cross-node communication.
- Expert capacity factor: buffer size for handling uneven routing. Too small = dropped tokens. Too large = wasted memory.

**Fault tolerance (critical for this scale):**
- MTBF at 4096 GPUs: approximately 1 failure every 2-4 hours
- Checkpoint every 10-15 minutes to distributed storage (async checkpointing to avoid stalling training)
- Elastic training: when a node fails, redistribute work to remaining nodes while replacement is provisioned
- Redundant networking: dual InfiniBand rails, automatic failover
- Health monitoring: per-GPU loss tracking, gradient norm monitoring, automatic detection of stuck/slow GPUs

**Monitoring dashboard:**
- Training loss curve (should be smooth; spikes indicate problems)
- MFU (Model FLOPs Utilization): typical MoE MFU is ~25-40% (dense models can reach 40-55%)
- GPU utilization, memory usage, communication overhead
- Expert utilization heatmap: are all experts being used?
- Checkpoint success rate, time-to-checkpoint

**Follow-up: "A GPU fails mid-training. Walk me through recovery."**
1. Detection: health monitor detects GPU timeout or NaN gradients (seconds)
2. The training process on the failed node crashes. Other nodes detect the communication failure.
3. Load latest checkpoint (stored on distributed filesystem, accessible from any node)
4. Reconfigure the process group: either use a spare node or redistribute using elastic training
5. Resume from checkpoint. Total downtime: 5-15 minutes.
6. Alert the ops team to replace the failed GPU.

Key insight: at this scale, failure handling is not an edge case -- it's a core design requirement. A training run that can't handle failures will never complete.

### Question 2 (45 min): "Design an inference system serving 100K QPS for a 70B model."

**Clarify requirements:**
- Model: 70B parameters
- QPS: 100K queries per second (very high -- this is a major product)
- Latency SLOs: p50 TTFT < 500ms, p99 TTFT < 2s, p50 generation < 50ms/token
- Average input: 200 tokens. Average output: 150 tokens.
- Budget: optimize for cost per request

**Back-of-envelope capacity planning:**
- 70B INT4 quantized: ~35GB, fits on 1 H100
- Per-GPU throughput with continuous batching: ~150 tok/s output at batch size 32
- Each request generates ~150 tokens. Time per request: ~1s.
- At batch size 32: ~32 QPS per GPU
- For 100K QPS: need ~3125 GPUs. Round to ~3200 H100s (400 nodes).
- Cost: 3200 x $3/hr = $9600/hr = ~$7M/month. Cost per request: ~$0.003.

**Architecture:**

```
[Load Balancer (L7)]
        |
[Request Router -- routes by prefix hash for cache hits]
        |
[Inference Worker Pool: 400 nodes x 8 GPUs]
   |-- vLLM with PagedAttention
   |-- INT4 quantization (AWQ or GPTQ)
   |-- Continuous batching
   |-- Prefix caching (KV cache reuse for common system prompts)
        |
[Output Streaming via SSE/WebSocket]
```

**Key optimizations:**
1. **PagedAttention (vLLM):** Manages KV cache like virtual memory. Eliminates memory waste from pre-allocated KV slots. Increases throughput 2-4x.
2. **Continuous batching:** Don't wait for a batch to fill. Add new requests to the running batch as slots free up. Critical for latency at scale.
3. **Prefix caching:** If many requests share the same system prompt (common in API deployments), cache the KV states for the shared prefix. Saves 30-70% of prefill compute.
4. **INT4 quantization:** 2x memory reduction, ~1.5x throughput improvement with <1% quality loss for a 70B model.
5. **Speculative decoding:** Use a small draft model (7B) to propose tokens, verify in parallel with the large model. 2-3x speedup for generation.

**Autoscaling:**
- Primary signal: queue depth (requests waiting for a GPU slot)
- Scale up: when average queue depth > 10 for 30 seconds
- Scale down: when GPU utilization < 30% for 5 minutes
- Keep 10-15% headroom for traffic spikes

**Monitoring:**
- TTFT (time to first token), TPS (tokens per second), p50/p95/p99
- GPU memory utilization, KV cache hit rate
- Error rate, timeout rate
- Cost per request

**Follow-up: "A customer wants guaranteed 200ms latency. Is that possible? How?"**

Be honest: 200ms for 150 output tokens is not possible with autoregressive generation (that would require 750 tok/s per request). But 200ms TTFT is achievable:
- Dedicated GPU pool (no sharing, no queuing)
- Speculative decoding for faster generation
- Smaller quantized model variant
- Geographic routing to minimize network latency
- The real answer: negotiate the SLO. 200ms TTFT + streaming output is realistic. 200ms total latency for 150 tokens is not.

---

### Question 3 (45 min): "Design the RLHF training pipeline -- not the algorithm, the SYSTEM."

**What the interviewer is testing:** Can you reason about the system-level challenges of RLHF that go beyond the math?

**The 4-model memory challenge:**
RLHF requires 4 models in memory simultaneously:
1. **Policy model** (the model being trained): 70B = 140GB fp16
2. **Reference model** (frozen copy for KL penalty): 70B = 140GB fp16
3. **Reward model** (scores generations): 70B = 140GB fp16
4. **Value model** (critic for PPO): 70B = 140GB fp16

Total: ~560GB just for model weights. Plus optimizer states, activations, KV cache for generation.

**The generation bottleneck:**
- RLHF requires generating completions from the policy model (to get rewards and train on)
- Generation is autoregressive and slow: 80-90% of RLHF training time is spent on generation
- This is the key system bottleneck

**Architecture:**

```
[Prompt Dataset]
       |
[Generation Engine (vLLM)]  <-- 80-90% of time spent here
   |-- Policy model loaded in inference mode
   |-- Continuous batching for throughput
   |-- Generate 256-512 token completions
       |
[Reward Scoring Engine]
   |-- Distributed reward model
   |-- Batch scoring of generations
       |
[PPO Training Engine (DeepSpeed/FSDP)]
   |-- Policy + Value model training
   |-- Reference model for KL penalty
   |-- Standard distributed training
       |
[KL Monitoring Dashboard]
   |-- Track KL divergence from reference
   |-- Alert if KL exceeds threshold
   |-- Reward distribution monitoring
```

**Key system decisions:**
1. **Async pipeline:** Generation and training can be overlapped. While batch N is training, batch N+1 is generating. This hides the generation latency.
2. **Separate GPU pools:** Use inference-optimized GPUs for generation (quantized, vLLM) and training-optimized GPUs for PPO. Different optimization profiles.
3. **Reference model sharing:** The reference model is frozen. Deploy it as a shared inference service rather than loading a copy per training worker.
4. **Reward model batching:** Score all generations in a large batch for efficiency. The reward model is also frozen, so deploy as inference service.
5. **Checkpointing and rollback:** RLHF can diverge. Save checkpoints frequently and monitor reward + KL. If KL spikes or reward collapses, roll back to last good checkpoint.

**Monitoring dashboard (what you actually watch):**
- Reward distribution: should shift right over training. If it plateaus or collapses, something is wrong.
- KL divergence from reference: should increase slowly. If it spikes, the model is diverging.
- Generation length distribution: reward hacking often manifests as length gaming.
- PPO metrics: value loss, policy loss, clip fraction, entropy.
- Sample outputs: periodically inspect actual model outputs for quality regression.

### Production Notes: System Design Reality Check

*Numbers and details that demonstrate production experience.*

- **Hardware reality:**
  - H100: 80GB HBM3, 3.35TB/s memory bandwidth, ~$30K purchase / ~$3/hr cloud
  - A single 70B fp16 model: ~140GB. Needs minimum 2 H100s just for weights.
  - INT4 quantized 70B: ~35GB. Fits on 1 H100 with room for KV cache.
  - NVLink within node: 900 GB/s. InfiniBand across nodes: 400 Gbps (50 GB/s). This 18x difference is why TP stays within the node.

- **Training reality:**
  - Llama 3 405B: 16,384 H100s, 54 days, estimated $100M+ total cost
  - GPU failure at scale: at 10K GPUs, expect a failure every 30-60 minutes. Checkpointing is not optional.
  - MFU (Model FLOPs Utilization): 30-45% is typical. 50%+ is excellent. This means more than half the theoretical compute is wasted on communication, memory operations, and idle time.

- **Inference reality:**
  - Production systems: vLLM (open source) or TensorRT-LLM (NVIDIA) or custom solutions
  - Typical production throughput: 50-200 tok/s per H100 for 70B (highly batch-size dependent)
  - KV cache is the memory bottleneck: a single request with 4K context on a Llama-2-70B-class model (GQA, 8 KV heads) uses ~1.3GB of KV cache in fp16 -- and forgetting GQA inflates that estimate ~8x (to ~10.7GB), a classic interview trap
  - Prefix caching saves 30-70% of prefill compute for API deployments with common system prompts

- **Interview timing discipline:**
  - You get 45 minutes total. Do NOT spend 20 minutes on requirements.
  - 5 min requirements, 25 min architecture (this is where you win), 10 min deep dive into one component, 5 min monitoring/cost.
  - Always do a back-of-envelope calculation. Interviewers love seeing you work with real numbers.
  - Name specific technologies (vLLM, DeepSpeed, FSDP, InfiniBand). Vague answers ("we'd use some parallelism") are a red flag.

---
## 12. Flashcard Summary

**Instructions**: Cover the answer column. Test yourself. If you can't answer in 10 seconds, mark it for review.

| # | Question | Answer |
|---|----------|--------|
| 1 | What is DDP? | Distributed Data Parallel: replicate full model on each GPU, split data, AllReduce gradients. |
| 2 | What is FSDP? | Fully Sharded Data Parallel: shard params + grads + optimizer across GPUs. AllGather before each layer. |
| 3 | What are ZeRO Stage 1/2/3? | Stage 1: shard optimizer. Stage 2: shard optimizer + gradients. Stage 3: shard everything (= FSDP). |
| 4 | What is tensor parallelism? | Split individual weight matrices across GPUs (column-parallel or row-parallel). Best with fast intra-node interconnect (NVLink). |
| 5 | What is pipeline parallelism? | Split layers across GPUs sequentially. Use micro-batch scheduling (1F1B) to reduce bubble time. |
| 6 | What is expert parallelism? | For MoE models: place different experts on different GPUs. All-to-all communication for token routing. |
| 7 | What is KV-cache? | Store past key/value pairs during autoregressive generation to avoid recomputing K/V for past tokens. Attention for each new token still reads the whole cache, so per-token attention cost remains O(n). |
| 8 | What is PagedAttention? | Treat KV-cache like OS virtual memory pages. Eliminates fragmentation, enables near-zero waste. 2-4x throughput vs serious serving systems (the famous 24x was vs naive HuggingFace Transformers). |
| 9 | What is continuous batching? | Add new requests as old ones finish, instead of waiting for the entire batch. No wasted compute on padding. |
| 10 | What is speculative decoding? | Use small draft model to guess N tokens, verify with large model in one pass. No quality loss, 2-6x speedup. |
| 11 | Name 3 quantization methods. | GPTQ (post-training weight quant), AWQ (activation-aware), GGUF (CPU-friendly format). |
| 12 | What is prefix caching? | Cache KV pairs for shared prompt prefixes (system prompts, few-shot examples) and reuse across requests. |
| 13 | How much memory for 70B fp16 + Adam? | Params: 140GB + Grads: 140GB + Optimizer: 840GB = 1120GB (1.12TB) total. |
| 14 | How many GPUs for 70B training with ZeRO-3? | 70B with mixed-precision Adam ~ 16 bytes/param ~ 1,120GB of state. Over 8 GPUs = 140GB/GPU -- does NOT fit 80GB A100s. Need more GPUs (e.g., 16+ -> ~70GB/GPU) and/or offload + activation tradeoffs. (112GB/8 = 14GB is the arithmetic for a 7B model.) |
| 15 | What are the 4 models in PPO-based RLHF? | Policy (trained), Reference (frozen, KL anchor), Reward Model (frozen), Value Model (trained). |
| 16 | What is the bottleneck in online RLHF? | Generation: autoregressive sampling from the policy model is slow. Solution: separate generation cluster or async pipeline. |
| 17 | How to load-balance LLM inference? | Least-queue-depth or least-connections (not round-robin). Consider input length in routing decisions. |
| 18 | What SLOs matter for LLM serving? | TTFT (time to first token), ITL (inter-token latency), throughput (QPS), error rate. |
| 19 | Pipeline parallel bubble fraction? | Utilization = m / (m + p - 1) for p stages, m micro-batches (e.g., p=4, m=4 -> 57%); bubble overhead relative to compute time is (p-1)/m. Increase micro-batches to reduce it. |
| 20 | 7-step ML system design framework? | Requirements, Architecture, Data Flow, Training Pipeline, Serving, Monitoring, Cost. |

---
## 13. Paper Guide: PagedAttention / vLLM

### Paper: "Efficient Memory Management for Large Language Model Serving with PagedAttention"

**Link**: [https://arxiv.org/abs/2309.06180](https://arxiv.org/abs/2309.06180)

### Why Read This Paper

This is the foundational paper behind vLLM, the most widely used open-source LLM serving framework. If you're asked about inference optimization at any frontier lab, you need to know this cold.

### Key Contributions

1. **Identified the KV-cache memory management problem**: Prior systems wasted 60-80% of KV-cache memory due to fragmentation.
2. **PagedAttention algorithm**: Inspired by OS virtual memory, breaks KV-cache into fixed-size pages.
3. **Block-level memory management**: Page table maps logical blocks to physical blocks, enabling non-contiguous storage.
4. **Copy-on-write**: For beam search and parallel sampling, share KV blocks until they diverge.

### Reading Strategy (2-hour deep read)

| Section | Time | Focus On |
|---------|------|----------|
| Abstract + Intro | 15 min | Problem statement, key insight |
| Section 2 (Background) | 15 min | How KV-cache works, memory waste analysis |
| Section 3 (PagedAttention) | 30 min | The core algorithm, page table, block allocation |
| Section 4 (System Design) | 20 min | Scheduler, block manager, how it integrates with the model |
| Section 5 (Evaluation) | 20 min | Throughput numbers, memory efficiency, comparison to baselines |
| Figures 3-8 | 20 min | Visual understanding of paging, block sharing, scheduling |

### Questions to Answer After Reading

1. Why can't we just pre-allocate KV-cache for max sequence length? What's the waste?
2. How does PagedAttention handle variable-length sequences in a batch?
3. What's the overhead of the page table lookup? Is it significant?
4. How does copy-on-write work for beam search? Why is it important?
5. What are the limitations of PagedAttention? When might it not help?

### Connection to Interview

When asked "Design an LLM serving system," you should be able to:
- Explain WHY KV-cache management matters (it's often the memory bottleneck)
- Describe HOW PagedAttention works (page table, block allocation, copy-on-write)
- Quote NUMBERS (2-4x throughput improvement, near-zero fragmentation waste)
- Discuss ALTERNATIVES (prefix caching, speculative decoding) and how they compose

---

**End of Notebook 21. Next: Notebook 22 (Research Presentation Guide).**

---

### Additional Reference: OpenRLHF (2024)

**Link**: [https://github.com/OpenRLHF/OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

**What it is**: An open-source framework for distributed RLHF training. Uses Ray + vLLM + DeepSpeed ZeRO-3 to orchestrate the 4-model RLHF pipeline (policy, reference, reward, value) across GPU clusters.

**Why it matters for interviews**: It's a concrete, open implementation of the exact RLHF training pipeline described in Practice Problem 1. If asked 'How would you implement distributed RLHF?', being able to reference OpenRLHF's architecture (Ray for orchestration, vLLM for fast generation, ZeRO-3 for memory efficiency) shows practical awareness beyond theory.

**Key design choices**: Separates generation (vLLM, inference-optimized) from training (DeepSpeed, gradient-optimized). Uses Ray for cross-node coordination. Supports PPO, DPO, GRPO, and rejection sampling workflows.